# Which GPU work did PyTorch launch?

`capture` returns `prof.events()` unchanged. Locally, `rows` keeps every event
once, in its original order, with its original unique row index.
Object references (`cpu_parent`, `cpu_children`, `kernels`) are replaced by
`launcher_id`, which points to another row in this same table:

- CPU event: its PyTorch `cpu_parent` row index.
- GPU kernel/copy/memset: the CUDA runtime or driver row matched by
  `correlation_id`; if no unique launch row exists, fall back to the CPU frontend
  event matched by `linked_correlation_id`.
- Other device events: use their `linked_correlation_id`, when available.

Follow `launcher_id` to trace any kernel back through the CPU operations that
led to it, or filter on `launcher_id` to inspect an event's direct children.
The resulting chain includes the launch API call, for example
`aten::mm -> cudaLaunchKernel -> kernel`. No execution groups are created.
Frontend and runtime/driver correlation lookups are kept separate within each
`node_id`. Missing or ambiguous matches use the frontend fallback where possible;
otherwise the link stays null. All events are retained.


In [1]:
import torch
from models.transformer import ModelParameters, build

# Built on CPU; capture moves it to the GPU on Modal.
model_parameters = ModelParameters(
    vocab_size=1000, sequence_length=64, num_layers=5,
    d_model=64, d_ff=64, num_heads=2, rope_theta=10000,
    device="cpu", dtype="torch.float32",
)
torch.manual_seed(0)
model = build(model_parameters)
tokens = torch.randint(
    0, model_parameters.vocab_size, (8, model_parameters.sequence_length),
    generator=torch.Generator().manual_seed(0),
)

In [2]:
from profile_events import app, image


@app.function(image=image, gpu="T4", timeout=300)
def capture(model, tokens) -> list:
    import torch
    from torch import nn
    from torch.profiler import ProfilerActivity, profile, record_function

    torch.manual_seed(0)
    model = model.to("cuda")
    tokens = tokens.to("cuda")
    model.train()
    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)
    # Next-token prediction: each position predicts the following input token.
    targets = tokens[:, 1:].reshape(-1)

    def train_step():
        with record_function("train::zero_grad"):
            optimizer.zero_grad(set_to_none=True)
        with record_function("train::forward"):
            logits = model(tokens)
        with record_function("train::loss"):
            loss = nn.functional.cross_entropy(
                logits[:, :-1, :].reshape(-1, logits.shape[-1]), targets
            )
        with record_function("train::backward"):
            loss.backward()
        with record_function("train::optimizer_step"):
            optimizer.step()

    with profile(
        activities=[ProfilerActivity.CPU, ProfilerActivity.CUDA],
        record_shapes=True,
        with_stack=False,
        with_flops=True,
        profile_memory=True,
    ) as prof:
        train_step()
        # Kernels run asynchronously. Wait for them inside the profiler so every launched
        # activity has finished, and been recorded, before collection stops.
        torch.cuda.synchronize()
    # The FunctionEvents themselves: they pickle with their cpu_parent/cpu_children references.
    return list(prof.events())

In [3]:
with app.run():
    function_events = capture.remote(model, tokens)
print(f"{len(function_events)} events")

9151 events


In [104]:
import pandas as pd


def _cpu_correlations(events, activity_types):
    """Map (node, correlation) to a CPU row ID; ambiguous matches stay None."""
    lookup = {}
    for event in events:
        if not event.is_cpu or event.activity_type not in activity_types:
            continue
        if pd.isna(event.correlation_id) or event.correlation_id == 0:
            continue
        key = (event.node_id, event.correlation_id)
        lookup[key] = event.Index if key not in lookup else None
    return lookup


def _validate_launchers(parents, cpu_ids):
    """Reject missing/non-CPU parents and cycles in O(n), without recursion."""
    for index, parent in parents.items():
        if parent is not None and parent not in cpu_ids:
            raise ValueError(f"Event {index} refers to missing or non-CPU launcher {parent}")

    checked = set()
    for index in parents:
        current, visiting = index, set()
        while current is not None and current not in checked:
            if current in visiting:
                raise ValueError("Launcher links contain a cycle")
            visiting.add(current)
            current = parents[current]
        checked.update(visiting)


def link_events(frame):
    """Connect events to their immediate CPU origin using original row IDs.

    ID reference
    ------------
    frame.index
        Unique integer row ID assigned when building the original event table.
        This is the identity used for navigation with frame.loc[row_id]. It is
        NOT a profiler correlation ID. Preserve it when filtering the table.
    launcher_id
        A row ID in this same table, or NA when no parent is known. On input,
        CPU rows must contain their PyTorch cpu_parent converted from an object
        reference to its row ID. On output, CPU parents are preserved and GPU
        rows point to their matched CPU launch call or frontend operation.
        For a CPU row, "launcher" means its enclosing CPU parent scope; it
        does not imply that every CPU event launches GPU work.
    correlation_id
        The original FunctionEvent.id, renamed during table construction.
        CPU frontend operations (cpu_op) and scopes (user_annotation) use
        PyTorch's counter. CUDA runtime/driver calls and GPU work use CUPTI's
        counter: a GPU activity shares its launch call's correlation ID.
        These counters can collide, so frontend and launch lookups are kept
        separate. A correlation ID is not a unique event-row identifier.
    linked_correlation_id
        For CUDA runtime/driver calls and GPU work, identifies the originating
        CPU frontend operation's correlation_id. It is not that operation's
        row ID, and does not identify the intermediate CUDA launch row.
        Zero or missing values mean there is no usable correlation link.
    node_id
        Profiler node identifier used to keep correlations from different
        nodes separate. Every lookup uses (node_id, correlation_id); absent
        or missing node IDs use -1. This is neither a row ID nor a GPU ID.
    external_id
        Raw profiler external-correlation metadata, retained unchanged if
        present. It is not a row ID and this function does not use it.

    Linking rules
    -------------
    1. CPU events keep the cpu_parent row links already computed by PyTorch.
    2. GPU kernels, copies and memsets first match correlation_id against CPU
       cuda_runtime/cuda_driver rows, inserting the CUDA API call in the path.
    3. If there is no unique launch match, match linked_correlation_id against
       CPU cpu_op/user_annotation rows instead. Other device events only use
       this fallback; GPU annotation spans must not join unrelated launch IDs.

    Missing or duplicate correlation matches are unresolved, never selected
    arbitrarily. After fallback, unresolved events retain launcher_id=NA.
    No time-based parent inference or cross-thread CPU ancestry is added.

    For example, following launcher_id from a kernel might visit original row
    IDs 50 -> 40 -> 30 -> 20, representing:
        GPU kernel -> cudaLaunchKernel -> aten::mm -> enclosing CPU scope
    Read the path in reverse for execution provenance. To find direct children
    of row 30, use rows.loc[rows["launcher_id"].eq(30)].

    Returns a new DataFrame in O(n) time and space, preserving every row, its
    index, order and other columns. Only launcher_id is updated (nullable
    Int64); no groups or duplicated ancestor rows are created. Raises
    ValueError for non-unique row IDs, missing/non-CPU parents, or cycles.
    """
    if not frame.index.is_unique:
        raise ValueError("Event row indices must be unique")

    # Only read linking fields, leaving large metadata (stacks, shapes) untouched.
    columns = ["activity_type", "correlation_id", "linked_correlation_id", "launcher_id"]
    events = list(frame[columns].assign(
        is_cpu=frame["device_type"].map(str).eq("DeviceType.CPU"),
        node_id=frame.get("node_id", pd.Series(-1, index=frame.index)).fillna(-1),
    ).itertuples())

    # PyTorch frontend IDs and CUPTI launch IDs are separate namespaces.
    frontend = _cpu_correlations(events, {"cpu_op", "user_annotation"})
    launches = _cpu_correlations(events, {"cuda_runtime", "cuda_driver"})
    gpu_work = {"kernel", "gpu_memcpy", "gpu_memset"}

    parents = {}
    for event in events:
        if event.is_cpu:
            parents[event.Index] = int(event.launcher_id) if pd.notna(event.launcher_id) else None
            continue

        launcher = None
        # Annotation spans must not match unrelated CUDA launch IDs.
        if event.activity_type in gpu_work and pd.notna(event.correlation_id):
            launcher = launches.get((event.node_id, event.correlation_id))
        if launcher is None and pd.notna(event.linked_correlation_id):
            launcher = frontend.get((event.node_id, event.linked_correlation_id))
        parents[event.Index] = launcher

    _validate_launchers(parents, {event.Index for event in events if event.is_cpu})
    return frame.assign(launcher_id=pd.array(list(parents.values()), dtype="Int64"))


In [105]:
position = {id(event): i for i, event in enumerate(function_events)}
raw_rows = (
    pd.DataFrame([vars(event) for event in function_events])
    .rename(columns={"id": "correlation_id"})
    .drop(columns=["cpu_parent", "cpu_children", "kernels"])
    .assign(launcher_id=pd.array(
        [position.get(id(event.cpu_parent)) for event in function_events], dtype="Int64"))
    .rename_axis("index")
)
rows = link_events(raw_rows)
# Every original event stays at the same unique row index.
assert rows.index.equals(raw_rows.index)

# Every kernel execution, including those without a known launcher.
kernel_ids = rows.index[rows["activity_type"].eq("kernel")].tolist()
kernels = rows.loc[kernel_ids]


## What each id is

`correlation_id` is `FunctionEvent.id`, which `_parse_kineto_results` fills with
`kineto_event.correlation_id()`. What it counts depends on the kind of row:

| Row (`activity_type`) | `correlation_id` | `linked_correlation_id` | `external_id` |
| --- | --- | --- | --- |
| Frontend: op or `record_function` scope (`cpu_op`, `user_annotation`) | PyTorch's own counter | 0 | `correlation_id` |
| Launch call on the CPU (`cuda_runtime`, `cuda_driver`) | CUPTI's counter | the op's `correlation_id` | `linked_correlation_id` |
| GPU work (`kernel`, `gpu_memcpy`, `gpu_memset`) | CUPTI's counter, equal to its launch call's | the op's `correlation_id` | `linked_correlation_id` |
| GPU span of a scope (`gpu_user_annotation`) | its scope's `correlation_id` | 0 | `correlation_id` |
| Profiler overhead (`overhead`) | reuses the `correlation_id` of the op it interrupted | 0 | 0 |

Two counters share the `correlation_id` column, so every lookup by it is restricted to one kind of row:
in one capture `Optimizer.step`'s GPU span and an unrelated launch call were both 1912.
`external_id` holds no value of its own: `KinetoEvent::externalId()` returns
`linked_correlation_id` when there is one, else `correlation_id`.

In [109]:
kernels

,correlation_id,node_id,name,overload_name,trace_name,time_range,thread,fwd_thread,count,input_shapes,...,external_id,linked_correlation_id,event_metadata,structured_input_shapes,structured_input_strides,input_dtypes,python_id,python_parent_id,python_module_id,launcher_id
index,,,,,,,,,,,,,,,,,,,,,
11,13,-1,"void at::native::vectorized_gather_kernel<16, ...",,"void at::native::vectorized_gather_kernel<16, ...",<torch.autograd.profiler_util.Interval object ...,1,0,1,[],...,4,4,None,[],[],[],-1,-1,-1,8
18,35,-1,void (anonymous namespace)::elementwise_kernel...,,void (anonymous namespace)::elementwise_kernel...,<torch.autograd.profiler_util.Interval object ...,1,0,1,[],...,10,10,None,[],[],[],-1,-1,-1,16
25,48,-1,void at::native::vectorized_elementwise_kernel...,,void at::native::vectorized_elementwise_kernel...,<torch.autograd.profiler_util.Interval object ...,1,0,1,[],...,13,13,None,[],[],[],-1,-1,-1,23
29,65,-1,"void at::native::reduce_kernel<512, 1, at::nat...",,"void at::native::reduce_kernel<512, 1, at::nat...",<torch.autograd.profiler_util.Interval object ...,1,0,1,[],...,16,16,None,[],[],[],-1,-1,-1,27
33,78,-1,void at::native::vectorized_elementwise_kernel...,,void at::native::vectorized_elementwise_kernel...,<torch.autograd.profiler_util.Interval object ...,1,0,1,[],...,17,17,None,[],[],[],-1,-1,-1,31
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8890,16716,-1,void at::native::(anonymous namespace)::multi_...,,void at::native::(anonymous namespace)::multi_...,<torch.autograd.profiler_util.Interval object ...,1,0,1,[],...,7298,7298,None,[],[],[],-1,-1,-1,8888
9134,17061,-1,void at::native::(anonymous namespace)::multi_...,,void at::native::(anonymous namespace)::multi_...,<torch.autograd.profiler_util.Interval object ...,1,0,1,[],...,7491,7491,None,[],[],[],-1,-1,-1,9132
9138,17070,-1,void at::native::(anonymous namespace)::multi_...,,void at::native::(anonymous namespace)::multi_...,<torch.autograd.profiler_util.Interval object ...,1,0,1,[],...,7540,7540,None,[],[],[],-1,-1,-1,9136
